# 01 — Source data audit

This notebook establishes the raw-table grains, population coverage, and temporal rules before feature engineering. The target is Home Credit's **payment-difficulty outcome**, not a regulatory default definition. No model decisions are made here.

In [1]:
from pathlib import Path
import gc
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.data import validate_raw_files, load_application
from src.validation import plot_target_distribution

RAW = ROOT / "data" / "raw"
TABLES = ROOT / "outputs" / "tables"
FIGURES = ROOT / "outputs" / "figures"
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

inventory = validate_raw_files(RAW)
application = load_application(RAW)
train_ids = set(application["SK_ID_CURR"])
inventory

,file,bytes,columns
0,application_train.csv,166133370,122
1,HomeCredit_columns_description.csv,37383,5
2,bureau.csv,170016717,17
3,bureau_balance.csv,375592889,3
4,previous_application.csv,404973293,37
5,installments_payments.csv,723118349,8


## Grain and temporal checks

The historical cutoff is the current application date (day/month 0). Bureau records require both origination and last-update dates at or before that cutoff. Previous applications must precede day 0. Installments require both scheduled and actual payment dates to be present and no later than day 0. Bureau balance months must be non-positive.

In [2]:
specs = [
    ("application_train.csv", ["SK_ID_CURR", "TARGET"], ["SK_ID_CURR"], None),
    ("HomeCredit_columns_description.csv", None, None, None),
    ("bureau.csv", ["SK_ID_CURR", "SK_ID_BUREAU", "DAYS_CREDIT", "DAYS_CREDIT_UPDATE"], ["SK_ID_BUREAU"], "DAYS_CREDIT / DAYS_CREDIT_UPDATE"),
    ("bureau_balance.csv", ["SK_ID_BUREAU", "MONTHS_BALANCE"], ["SK_ID_BUREAU", "MONTHS_BALANCE"], "MONTHS_BALANCE"),
    ("previous_application.csv", ["SK_ID_CURR", "SK_ID_PREV", "DAYS_DECISION"], ["SK_ID_PREV"], "DAYS_DECISION"),
    ("installments_payments.csv", ["SK_ID_CURR", "SK_ID_PREV", "NUM_INSTALMENT_VERSION", "NUM_INSTALMENT_NUMBER", "DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT"], ["SK_ID_CURR", "SK_ID_PREV", "NUM_INSTALMENT_VERSION", "NUM_INSTALMENT_NUMBER"], "DAYS_INSTALMENT / DAYS_ENTRY_PAYMENT"),
]
rows = []
bureau_map = None
for filename, usecols, keys, temporal_field in specs:
    encoding = "latin1" if filename.startswith("HomeCredit") else "utf-8"
    frame = pd.read_csv(RAW / filename, usecols=usecols, encoding=encoding)
    row = {
        "source": filename,
        "row_count": len(frame),
        "column_count": int(inventory.loc[inventory["file"].eq(filename), "columns"].iloc[0]),
        "grain_key": "+".join(keys) if keys else "data dictionary row",
        "missing_key_rows": int(frame[keys].isna().any(axis=1).sum()) if keys else 0,
        "duplicate_grain_rows": int(frame.duplicated(keys, keep=False).sum()) if keys else 0,
        "temporal_field": temporal_field or "not applicable",
        "temporal_min": None,
        "temporal_max": None,
        "temporal_excluded_rows": 0,
        "coverage_rate": None,
        "coverage_definition": "not applicable",
    }
    if filename == "bureau.csv":
        eligible = frame["DAYS_CREDIT"].le(0) & frame["DAYS_CREDIT_UPDATE"].le(0)
        row.update(temporal_min=float(frame[["DAYS_CREDIT", "DAYS_CREDIT_UPDATE"]].min().min()), temporal_max=float(frame[["DAYS_CREDIT", "DAYS_CREDIT_UPDATE"]].max().max()), temporal_excluded_rows=int((~eligible).sum()), coverage_rate=frame.loc[frame["SK_ID_CURR"].isin(train_ids), "SK_ID_CURR"].nunique() / len(train_ids), coverage_definition="train borrowers represented / application borrowers")
        bureau_map = frame.loc[eligible & frame["SK_ID_CURR"].isin(train_ids), ["SK_ID_BUREAU", "SK_ID_CURR"]]
    elif filename == "bureau_balance.csv":
        mapped_borrowers = bureau_map.loc[bureau_map["SK_ID_BUREAU"].isin(set(frame["SK_ID_BUREAU"])), "SK_ID_CURR"].nunique()
        row.update(temporal_min=float(frame["MONTHS_BALANCE"].min()), temporal_max=float(frame["MONTHS_BALANCE"].max()), temporal_excluded_rows=int(frame["MONTHS_BALANCE"].gt(0).sum()), coverage_rate=mapped_borrowers / len(train_ids), coverage_definition="train borrowers represented through eligible bureau records / application borrowers")
    elif filename == "previous_application.csv":
        row.update(temporal_min=float(frame["DAYS_DECISION"].min()), temporal_max=float(frame["DAYS_DECISION"].max()), temporal_excluded_rows=int(frame["DAYS_DECISION"].ge(0).sum()), coverage_rate=frame.loc[frame["SK_ID_CURR"].isin(train_ids), "SK_ID_CURR"].nunique() / len(train_ids), coverage_definition="train borrowers represented / application borrowers")
    elif filename == "installments_payments.csv":
        valid = frame["DAYS_INSTALMENT"].notna() & frame["DAYS_ENTRY_PAYMENT"].notna() & frame["DAYS_INSTALMENT"].le(0) & frame["DAYS_ENTRY_PAYMENT"].le(0)
        row.update(temporal_min=float(frame[["DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT"]].min().min()), temporal_max=float(frame[["DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT"]].max().max()), temporal_excluded_rows=int((~valid).sum()), coverage_rate=frame.loc[frame["SK_ID_CURR"].isin(train_ids), "SK_ID_CURR"].nunique() / len(train_ids), coverage_definition="train borrowers represented / application borrowers")
    rows.append(row)
    del frame
    gc.collect()

dataset_summary = pd.DataFrame(rows)
dataset_summary.to_csv(TABLES / "dataset_summary.csv", index=False)
dataset_summary

,source,row_count,column_count,grain_key,missing_key_rows,duplicate_grain_rows,temporal_field,temporal_min,temporal_max,temporal_excluded_rows,coverage_rate,coverage_definition
0,application_train.csv,307511,122,SK_ID_CURR,0,0,not applicable,NaN,NaN,0,NaN,not applicable
1,HomeCredit_columns_description.csv,219,5,data dictionary row,0,0,not applicable,NaN,NaN,0,NaN,not applicable
2,bureau.csv,1716428,17,SK_ID_BUREAU,0,0,DAYS_CREDIT / DAYS_CREDIT_UPDATE,-41947.0,372.0,17,0.856851,train borrowers represented / application borr...
3,bureau_balance.csv,27299925,3,SK_ID_BUREAU+MONTHS_BALANCE,0,0,MONTHS_BALANCE,-96.0,0.0,0,0.299927,train borrowers represented through eligible b...
4,previous_application.csv,1670214,37,SK_ID_PREV,0,0,DAYS_DECISION,-2922.0,-1.0,0,0.946493,train borrowers represented / application borr...
5,installments_payments.csv,13605401,8,SK_ID_CURR+SK_ID_PREV+NUM_INSTALMENT_VERSION+N...,0,1294388,DAYS_INSTALMENT / DAYS_ENTRY_PAYMENT,-4921.0,-1.0,2905,0.948399,train borrowers represented / application borr...


In [3]:
assert dataset_summary.loc[dataset_summary["source"].eq("application_train.csv"), "row_count"].iloc[0] == 307_511
assert dataset_summary.loc[dataset_summary["source"].eq("bureau.csv"), "temporal_excluded_rows"].iloc[0] == 17
assert dataset_summary.loc[dataset_summary["source"].eq("installments_payments.csv"), "temporal_excluded_rows"].iloc[0] == 2_905
plot_target_distribution(application["TARGET"], FIGURES / "target_distribution.png")
print(f"Applicants: {len(application):,}; events: {application['TARGET'].sum():,}; event rate: {application['TARGET'].mean():.3%}")

Applicants: 307,511; events: 24,825; event rate: 8.073%


### Audit conclusion

The application grain is unique. Bureau and previous-application identifiers are unique at their documented grains. Installment rows are intentionally non-unique at the scheduled-installment grain because partial payments can create several payment rows; feature engineering therefore consolidates payment rows before borrower aggregation. The explicit temporal exclusions above are carried into notebook 02.